# Phase 2 -- Graph + Features EDA

Checks the artifacts produced by `scripts/build_graph.py`:

* `data/graphs/hetero.pt` -- PyG HeteroData (7 node types, 8 edge types)
* `data/graphs/graph_builder.pkl` -- fitted graph builder state
* `data/graphs/features.parquet` -- 119 engineered columns + IDs
* `data/graphs/feature_pipeline.pkl` -- fitted feature pipeline

Run `python scripts/build_graph.py` (full set) or `python scripts/build_graph.py --nrows 100000` (subset) before opening this notebook.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    src = ROOT / "src"
elif (ROOT.parent / "src").exists():
    src = ROOT.parent / "src"
else:
    raise RuntimeError("Could not locate src/ relative to this notebook.")
sys.path.insert(0, str(src))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from fraud_detection.data.graph_builder import EDGE_TYPES, NODE_TYPES
from fraud_detection.features.pipeline import ALL_ENGINEERED_FEATURES
from fraud_detection.utils.config import load_config

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
cfg = load_config()
GRAPH_DIR = cfg.paths.data_graphs

## 1. HeteroData structure

In [ ]:
hetero_path = GRAPH_DIR / "hetero.pt"
if not hetero_path.exists():
    raise FileNotFoundError(f"{hetero_path} not found -- run `python scripts/build_graph.py` first.")
data = torch.load(hetero_path, weights_only=False)
print(data)
print()
print("Per-node stats:")
for nt in NODE_TYPES:
    print(f"  {nt:12s}  n={data[nt].num_nodes:>7}  feat_dim={data[nt].x.shape[1]}")
print()
print("Per-edge stats:")
for et in EDGE_TYPES:
    print(f"  {et[0]:>12s} -{et[1]:>15s}-> {et[2]:<12s}  edges={data[et].num_edges}")

## 2. Fraud rate by split

In [ ]:
y = data["transaction"].y.numpy()
rows = []
for mask_name in ("train_mask", "val_mask", "test_mask"):
    mask = getattr(data["transaction"], mask_name, None)
    if mask is None:
        continue
    m = mask.numpy().astype(bool)
    rows.append({"split": mask_name.replace("_mask", ""), "n": int(m.sum()), "fraud_rate": float(y[m].mean())})
pd.DataFrame(rows)

## 3. Engineered feature table

In [ ]:
features_path = GRAPH_DIR / "features.parquet"
features = pd.read_parquet(features_path)
print(f"Feature frame: {features.shape}")
print(f"Expected engineered columns: {len(ALL_ENGINEERED_FEATURES)}")
print(f"Total NaNs: {features.isna().sum().sum()}")
features.head(3).T

## 4. Top features by correlation with isFraud

In [ ]:
feat_cols = [c for c in features.columns if c.startswith("feat_")]
corrs = features[feat_cols].corrwith(features["isFraud"]).abs().sort_values(ascending=False)
top20 = corrs.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=top20.values, y=top20.index, ax=ax, palette="rocket")
ax.set_title("Top 20 |corr| of engineered features with isFraud")
ax.set_xlabel("|Pearson r|")
plt.tight_layout()
top20.to_frame("|corr|")

## 5. Graph-structural feature distributions

In [ ]:
gr_cols = [c for c in feat_cols if c.startswith("feat_gr_")]
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), gr_cols[:16]):
    sns.histplot(features[col], bins=40, ax=ax, color="#4c72b0")
    ax.set_title(col.replace("feat_gr_", ""), fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()

## 6. Fraud rate vs. ring membership

In [ ]:
if "feat_gr_ring_member" in features.columns:
    ring = features.groupby("feat_gr_ring_member")["isFraud"].agg(["size", "mean"])
    ring = ring.rename(columns={"size": "n", "mean": "fraud_rate"})
    print(ring)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(x=ring.index, y=ring["fraud_rate"], ax=ax, palette=["#4c72b0", "#c44e52"])
    ax.set_xlabel("ring_member")
    ax.set_ylabel("fraud rate")
    ax.set_title("Fraud rate by ring membership")
    plt.tight_layout()

## 7. Summary

In [ ]:
summary = {
    "n_transactions": int(data["transaction"].num_nodes),
    "n_cards": int(data["card"].num_nodes),
    "n_addresses": int(data["address"].num_nodes),
    "n_emails": int(data["email"].num_nodes),
    "n_devices": int(data["device"].num_nodes),
    "n_ips": int(data["ip_address"].num_nodes),
    "n_merchants": int(data["merchant"].num_nodes),
    "n_engineered_features": len(feat_cols),
    "total_edges": int(sum(data[et].num_edges for et in EDGE_TYPES)),
    "fraud_rate_overall": float(y.mean()),
}
pd.Series(summary, name="value").to_frame()